In [1]:
import pandas as pd
import os
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

root_path = os.path.dirname((os.getcwd()))

In [2]:
data_path = os.path.join(Path(os.getcwd()).resolve().parents[1], "dataset.xlsx")
df = pd.read_excel(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12095 entries, 0 to 12094
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   studentID         12095 non-null  object 
 1   classID           12095 non-null  object 
 2   timeStamp         12095 non-null  object 
 3   studentEmotion    12095 non-null  object 
 4   finalScore        12095 non-null  float64
 5   totalImages       12095 non-null  int64  
 6   learningTimes     12095 non-null  int64  
 7   finishedLession   12095 non-null  int64  
 8   avgTimeLearn      12095 non-null  float64
 9   avgTimeFinish     12095 non-null  float64
 10  percentageFinish  12095 non-null  float64
 11  timeInWeek        12095 non-null  float64
 12  inTime            12095 non-null  float64
 13  outTime           12095 non-null  float64
 14  inWeekday         12095 non-null  float64
 15  outWeekday        12095 non-null  float64
dtypes: float64(9), int64(3), object(4)
memor

In [3]:
df = df.sort_values(by=["studentID", "timeStamp"])

In [4]:
# group emotions into a "document" per student
docs = (
    df.groupby("studentID")["studentEmotion"]
      .apply(lambda x: " ".join(x.astype(str)))
)

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",  # keep single-word emotions
    lowercase=False
)

tfidf_matrix = vectorizer.fit_transform(docs)

In [5]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=docs.index,
    columns=vectorizer.get_feature_names_out()
)

In [6]:
tfidf_df

,happy,neutral,sad
studentID,,,
student_001,0.092949,0.841106,0.532824
student_002,0.000000,0.990096,0.140389
student_003,0.012045,0.399899,0.916480
student_004,0.066574,0.910152,0.408889
student_005,0.034871,0.976173,0.214175
...,...,...,...
student_165,0.000000,1.000000,0.000000
student_166,0.000000,1.000000,0.000000
student_167,0.438892,0.000000,0.898540


In [7]:
df = df.drop_duplicates(subset=["studentID"])

In [8]:
df_final = df.merge(tfidf_df, on="studentID", how="inner")

In [9]:
print(df_final.shape)
df_final.head(3)

(169, 19)


,studentID,classID,timeStamp,studentEmotion,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,timeInWeek,inTime,outTime,inWeekday,outWeekday,happy,neutral,sad
0,student_001,GENE1001-2-3-24(N01),2025-08-18 04:11:05.921000,neutral,10.0,263,214,87,12.7,31.2,49.1,2714.7,2033.4,681.3,2589.2,125.5,0.092949,0.841106,0.532824
1,student_002,GENE1001-2-3-24(N01),2025-08-18 05:19:40.974000,neutral,7.6,259,161,67,9.0,21.6,49.6,1450.5,1138.8,311.7,1229.4,221.1,0.000000,0.990096,0.140389
2,student_003,GENE1001-2-3-24(N01),2025-08-18 07:14:20.513000,sad,8.4,395,217,82,8.0,21.2,43.2,1740.6,702.4,1038.2,1404.2,336.4,0.012045,0.399899,0.916480


In [10]:
def log_transform(df, columns, eps=1e-6):
    df = df.copy()
    
    for col in columns:
        df[col] = np.log1p(np.clip(df[col], -1 + eps, None))
    
    return df

def z_score_transform(df, id_col):
    df = df.copy()
    cols_to_scale = df.columns.difference([id_col])
    scaler = StandardScaler()
    df.loc[:, cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
    return df

In [11]:
cols = df_final.drop(["studentID", "classID", "timeStamp", "finalScore", "studentEmotion"], axis=1).columns
master_df = log_transform(df=df_final, columns=cols)
master_df = master_df.drop(["classID", "timeStamp", "timeInWeek", "studentEmotion"], axis=1)
master_df = z_score_transform(df=master_df, id_col="studentID")

In [12]:
master_df.head(5)

,studentID,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday,happy,neutral,sad
0,student_001,0.743599,1.378500,2.034751,1.964586,0.593146,1.186554,-0.610426,1.948230,1.134181,2.041879,0.541544,0.013875,0.446626,0.247295
1,student_002,-0.318012,1.359847,1.611641,1.466515,0.098515,0.615339,-0.587080,1.441984,0.598606,1.338122,0.794792,-0.628991,0.828138,-1.013355
2,student_003,0.035858,1.873857,2.055466,1.851584,-0.067027,0.586525,-0.904990,1.020207,1.422990,1.463710,0.982919,-0.542394,-0.896551,1.199482
3,student_004,-0.141077,1.063510,1.305499,1.288070,1.365650,1.796760,-0.429843,0.641848,1.983156,1.712548,1.461780,-0.162813,0.627127,-0.112091
4,student_005,-1.025754,1.454741,1.361077,1.379352,-0.049666,0.274806,-0.500421,0.204625,1.268444,1.103607,0.606661,-0.381066,0.793716,-0.746105


In [ ]:
from sklearn.model_selection import train_test_split

In [14]:
target_col = "finalScore"
X = master_df.drop(columns=["studentID", "finalScore"], axis=1)
y = master_df[target_col]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.values
X_test = X_test.values

In [100]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, Reshape, Flatten
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

In [101]:
def transformer_encoder(x, head_size, num_heads, ff_dim, dropout=0.1):
    # Self-attention
    attn_output = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x, x)
    x = LayerNormalization(epsilon=1e-6)(x + attn_output)

    # Feed-forward network
    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dropout(dropout)(ff)
    ff = Dense(x.shape[-1])(ff)

    return LayerNormalization(epsilon=1e-6)(x + ff)


In [102]:
n_features = X_train.shape[1]

inputs = Input(shape=(n_features,))

# Convert features to tokens
x = Reshape((n_features, 1))(inputs)

# Linear embedding per feature
x = Dense(32)(x)   # embedding_dim = 32

# Transformer blocks
for _ in range(2):  # number of transformer layers
    x = transformer_encoder(
        x,
        head_size=32,
        num_heads=4,
        ff_dim=64,
        dropout=0.2
    )

# Flatten token representations
x = Flatten()(x)

# Regression head
x = Dense(32, activation="relu", kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)
x = Dense(16, activation="relu", kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.2)(x)

output = Dense(1, activation="linear")(x)

model = Model(inputs=inputs, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_5 (InputLayer)           [(None, 13)]         0           []                               
                                                                                                  
 reshape_4 (Reshape)            (None, 13, 1)        0           ['input_5[0][0]']                
                                                                                                  
 dense_32 (Dense)               (None, 13, 32)       64          ['reshape_4[0][0]']              
                                                                                                  
 multi_head_attention_8 (MultiH  (None, 13, 32)      16800       ['dense_32[0][0]',               
 eadAttention)                                                    'dense_32[0][0]']         

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",        
    patience=10,               
    min_delta=1e-4,            
    restore_best_weights=True, 
    verbose=1
)

In [104]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
5/5 [==============================] - 1s 60ms/step - loss: 3.2520 - mae: 1.2257 - val_loss: 1.2211 - val_mae: 0.7654
Epoch 2/100
5/5 [==============================] - 0s 13ms/step - loss: 1.6802 - mae: 0.9585 - val_loss: 1.0289 - val_mae: 0.5757
Epoch 3/100
5/5 [==============================] - 0s 13ms/step - loss: 1.4675 - mae: 0.7842 - val_loss: 1.0136 - val_mae: 0.5230
Epoch 4/100
5/5 [==============================] - 0s 14ms/step - loss: 1.2627 - mae: 0.7268 - val_loss: 0.9905 - val_mae: 0.5253
Epoch 5/100
5/5 [==============================] - 0s 22ms/step - loss: 1.1060 - mae: 0.6782 - val_loss: 0.9532 - val_mae: 0.5551
Epoch 6/100
5/5 [==============================] - 0s 17ms/step - loss: 1.1571 - mae: 0.6679 - val_loss: 0.9000 - val_mae: 0.5536
Epoch 7/100
5/5 [==============================] - 0s 14ms/step - loss: 0.9333 - mae: 0.5897 - val_loss: 0.8569 - val_mae: 0.5480
Epoch 8/100
5/5 [==============================] - 0s 21ms/step - loss: 0.9921 - mae: 0.63

In [105]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Predict
y_pred = model.predict(X_test).flatten()

# Metrics
mse  = mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("Evaluation Results (Transformer Regression)")
print(f"MSE  : {mse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

2/2 [==============================] - 0s 3ms/step
Evaluation Results (Transformer Regression)
MSE  : 0.5572
MAE  : 0.4784
R²   : 0.4502
